<a href="https://colab.research.google.com/github/ekrombouts/GenCareAI/blob/work_in_progress/scripts/work_in_progress/400_CarePlan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Summarisation: Data cleaning

**Author:** Eva Rombouts  
**Date:** 2024-07-21  
**Version:** 1.0

### Description
This script performs data cleaning on datasets of nursing home notes: ekrombouts/Galaxy_records and ekrombouts/Galaxy_summaries, containing summaries of client notes of fictional nursing home residents.
The cleaned dataset retains two columns: input and summary.

In [ ]:
!pip install -q datasets
from google.colab import drive, userdata
drive.mount('/content/drive')
HF_TOKEN = userdata.get('HF_TOKEN')

In [4]:
import pandas as pd
from datasets import load_dataset

In [5]:
dataset = load_dataset("ekrombouts/Galaxy_records", token=HF_TOKEN)
df_records = dataset['train'].to_pandas()
dataset = load_dataset("ekrombouts/Galaxy_summaries", token=HF_TOKEN)
df_summaries = dataset['train'].to_pandas()

README.md:   0%|          | 0.00/2.38k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/2.33M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/29668 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/2.42k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/638k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3396 [00:00<?, ? examples/s]

In [6]:
df = (
    df_records
    .groupby(['ct_id', 'month', 'iteration'])['note']
    .apply(lambda x: '\n- '.join(x))
    .reset_index()
    .assign(note=lambda df: '- ' + df['note'])
    .merge(df_summaries, on=['ct_id', 'month', 'iteration'], how='right')
    .assign(prev_summary=lambda df: df['summary'].shift(1))
    .dropna(subset=['note'])
    .assign(input=lambda df: df.apply(
        lambda row: f"{row['prev_summary']}\n\nRapportages:\n{row['note']}",
        axis=1
    ))
    [['input', 'summary']]
)

KeyError: 'month'

In [ ]:
sample = df.sample(2)

for s in sample.iterrows():
    print('---INPUT:---')
    print(s[1]['input'])
    print('\n---SAMENVATTING:---')
    print(s[1]['summary'])
    print('\n*************\n')
    print()

In [ ]:
df.to_csv('../data/galaxy_summaries.csv', index=False)